# Harmonization: UNSW-NB15 → ตารางกลาง

ใน notebook นี้แสดงการ map ข้อมูลไว้เป็นขั้น ๆ โดยตรง ไม่เรียก `harmonize_dataset()` หรือฟังก์ชัน schema จากไฟล์ภายนอก

เราอ่านผล cleansing ของ official Train/Test แล้วแยกเป็น `features`, `labels`, `metadata` ซึ่งใช้ `record_id` เชื่อมกัน คอลัมน์ที่หาไม่พบจะเป็น missing ไม่เดาจากตำแหน่ง ส่วน label ที่หายหรือไม่สอดคล้องจะทำให้หยุด

> แนะนำให้รัน EDA และ Cleansing ก่อน ตัวอย่าง output เป็น CSV และใช้เป็น input ของ `common/build_dataset.ipynb`

## 1. ตรวจผล cleansing, raw manifest และรุ่น contract

ยืนยันว่า raw ยังตรงกับ checksum ที่บันทึกตอน cleansing และได้ผล CSV ของ Train/Test ครบก่อนเริ่มแปลง เพื่อป้องกันการใช้ผลเก่ากับข้อมูลต้นฉบับคนละรุ่น

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
project_root = next((path for path in (current, *current.parents)
                     if (path / 'data-pipeline' / 'UNSW-NB15' / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('เปิด notebook จากภายในโฟลเดอร์โปรเจกต์ CSI_ML_Final')

pipeline_dir = project_root / 'data-pipeline'
dataset_dir = pipeline_dir / 'UNSW-NB15'
clean_dir = dataset_dir / 'output' / 'cleansing'
output_dir = dataset_dir / 'output' / 'harmonized'
split_files = {
    'train': clean_dir / 'UNSW_NB15_training-set.csv',
    'test': clean_dir / 'UNSW_NB15_testing-set.csv',
}
missing = [path.name for path in split_files.values() if not path.is_file()]
manifest_path = clean_dir / 'source_manifest.csv'
if missing or not manifest_path.is_file():
    raise FileNotFoundError(
        f'ไม่พบผล cleansing/manifest: {missing}; รัน UNSW-NB15/cleansing.ipynb ก่อน'
    )

def sha256_file(file_path):
    digest = hashlib.sha256()
    with file_path.open('rb') as file:
        for block in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

raw_dir = dataset_dir / 'raw'
raw_files = sorted(path for path in raw_dir.rglob('*')
                   if path.is_file() and path.suffix.lower() in {'.csv', '.gz', '.txt'})
current_manifest = pd.DataFrame([
    {'file': path.relative_to(raw_dir).as_posix(), 'bytes': path.stat().st_size,
     'sha256': sha256_file(path)} for path in raw_files
], columns=['file', 'bytes', 'sha256']).sort_values('file').reset_index(drop=True)
saved_manifest = pd.read_csv(manifest_path).sort_values('file').reset_index(drop=True)
if not current_manifest['file'].equals(saved_manifest['file']) or not current_manifest['sha256'].equals(saved_manifest['sha256']):
    raise ValueError('raw เปลี่ยนหลัง cleansing; รัน cleansing ใหม่ก่อน harmonization')

registry = json.loads((pipeline_dir / 'common' / 'dataset_registry.json').read_text(encoding='utf-8'))
contract = json.loads((pipeline_dir / 'common' / 'feature_contract.json').read_text(encoding='utf-8'))
print('ไฟล์พร้อม:', {name: path.name for name, path in split_files.items()})
print('raw checksum ตรงกับ manifest; dataset version:', registry['datasets']['UNSW-NB15']['version'])

## 2. กติกาแปลงที่ใช้กับทุกแถว

ค้นคอลัมน์จากชื่อ header หลัง normalize BOM/ช่องว่าง ไม่อาศัยลำดับคอลัมน์ จากนั้นแปลงค่าตัวเลข/หมวดหมู่ และเก็บค่าที่ใช้ไม่ได้เป็น missing พร้อม quality flag

In [ ]:
def find_column(frame, *candidate_names):
    normalized = {str(name).replace('\ufeff', '').strip().lower(): name
                  for name in frame.columns}
    for candidate in candidate_names:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    return None

def nonnegative_number(frame, column_name):
    if column_name is None or column_name not in frame.columns:
        empty_values = pd.Series(np.nan, index=frame.index, dtype='float64')
        no_invalid_values = pd.Series(False, index=frame.index, dtype=bool)
        return empty_values, no_invalid_values
    original = frame[column_name]
    number = pd.to_numeric(original, errors='coerce').astype('float64')
    invalid = original.notna() & (number.isna() | ~np.isfinite(number) | number.lt(0))
    number = number.replace([np.inf, -np.inf], np.nan)
    number.loc[number < 0] = np.nan
    return number, invalid.astype(bool)

def optional_number(frame, column_name):
    values, _ = nonnegative_number(frame, column_name)
    return values

def map_attack_category(raw_label, source_category):
    label = '' if pd.isna(raw_label) else str(raw_label).strip().lower()
    category = '' if pd.isna(source_category) else str(source_category).strip().lower()
    value = category or label
    if label in {'0', 'normal', 'benign', 'background'} or value in {'normal', 'benign', 'background'}:
        return 'Normal'
    category_map = {
        'dos': 'DoS/DDoS', 'ddos': 'DoS/DDoS',
        'reconnaissance': 'Probe/Recon', 'recon': 'Probe/Recon',
        'exploits': 'Exploit/U2R', 'exploit': 'Exploit/U2R', 'shellcode': 'Exploit/U2R',
        'backdoors': 'Botnet/Backdoor', 'backdoor': 'Botnet/Backdoor',
        'botnet': 'Botnet/Backdoor', 'worms': 'Botnet/Backdoor',
        'fuzzers': 'Generic/Fuzzer', 'fuzzer': 'Generic/Fuzzer',
        'generic': 'Generic/Fuzzer', 'analysis': 'Generic/Fuzzer',
    }
    return category_map.get(value, 'Other attack')

print('กติกาพร้อม: ชื่อคอลัมน์ → ค่ากลาง; label → taxonomy กลาง')

## 3. แปลงหนึ่ง split เป็น features, labels และ metadata

ตารางทั้งสามต้องมี record_id เดียวกันและเรียงเหมือนกัน `source_split` มาจากไฟล์ official ที่อ่านอยู่; `capture_time` แปลงจาก Unix seconds หากมี `stime`

In [ ]:
def harmonize_split(clean_frame, split_name, source_file):
    frame = clean_frame.copy()
    label_column = find_column(frame, 'label')
    category_column = find_column(frame, 'attack_cat')
    if label_column is None:
        raise ValueError(f'{source_file}: ไม่มีคอลัมน์ label')

    raw_label = frame[label_column].astype('string').str.strip()
    if raw_label.isna().any() or raw_label.eq('').any():
        raise ValueError(f'{source_file}: พบ label ว่าง; ไม่จัด label ที่หายเป็น attack')
    category = (frame[category_column].astype('string').str.strip()
                if category_column else pd.Series(pd.NA, index=frame.index, dtype='string'))
    original_binary = pd.to_numeric(raw_label, errors='coerce')
    if original_binary.isna().any() or not original_binary.isin([0, 1]).all():
        raise ValueError(f'{source_file}: label ต้องมีเฉพาะ 0/1')
    category_is_normal = category.str.lower().isin(['normal', 'benign'])
    if (category_is_normal.to_numpy() != original_binary.eq(0).to_numpy()).any():
        raise ValueError(f'{source_file}: label 0/1 ไม่ตรงกับ attack_cat normal/attack')
    attack_category = [map_attack_category(label, name)
                       for label, name in zip(raw_label, category)]

    duration, invalid_duration = nonnegative_number(frame, find_column(frame, 'dur'))
    forward_bytes, invalid_forward = nonnegative_number(frame, find_column(frame, 'sbytes'))
    backward_bytes, invalid_backward = nonnegative_number(frame, find_column(frame, 'dbytes'))
    protocol_column = find_column(frame, 'proto')
    if protocol_column:
        protocol = frame[protocol_column].astype('string').str.strip().str.lower()
        protocol = protocol.replace({'6': 'tcp', '17': 'udp', '1': 'icmp'})
        protocol = protocol.where(protocol.isna() | protocol.isin(['tcp', 'udp', 'icmp']), 'other')
    else:
        protocol = pd.Series(pd.NA, index=frame.index, dtype='string')

    service_column = find_column(frame, 'service')
    if service_column:
        service_text = frame[service_column].astype('string').str.strip().str.lower()
        service_groups = {'http':'http', 'www':'http', 'https':'https_ssl', 'ssl':'https_ssl',
                          'ftp':'ftp', 'ftp_data':'ftp', 'ssh':'ssh', 'smtp':'smtp',
                          'domain':'dns', 'domain_u':'dns', 'dns':'dns'}
        service = service_text.map(service_groups).fillna('other').where(service_text.notna())
    else:
        port_column = find_column(frame, 'Destination Port', 'dst port', 'dsport')
        if port_column:
            destination_port = pd.to_numeric(frame[port_column], errors='coerce')
            service = pd.Series('other', index=frame.index, dtype='string')
            port_groups = {80:'http', 443:'https_ssl', 53:'dns', 20:'ftp', 21:'ftp',
                           22:'ssh', 25:'smtp', 465:'smtp', 587:'smtp'}
            for port, group in port_groups.items():
                service.loc[destination_port.eq(port)] = group
            service = service.where(destination_port.notna(), pd.NA)
        else:
            service = pd.Series(pd.NA, index=frame.index, dtype='string')

    state_column = find_column(frame, 'state')
    if state_column:
        state_text = frame[state_column].astype('string').str.strip().str.lower()
        state_groups = {'sf':'established', 'fin':'established', 'con':'established',
                        's0':'no-reply', 'int':'no-reply', 'req':'no-reply',
                        'rej':'reset', 'rsto':'reset', 'rstr':'reset', 'rstos0':'reset', 'rst':'reset'}
        connection_state = state_text.map(state_groups).fillna('other').where(state_text.notna())
    else:
        connection_state = pd.Series(pd.NA, index=frame.index, dtype='string')

    packets_forward = optional_number(frame, find_column(frame, 'spkts'))
    packets_backward = optional_number(frame, find_column(frame, 'dpkts'))
    packet_total = packets_forward + packets_backward
    mean_packet_size = (forward_bytes + backward_bytes).div(packet_total.where(packet_total > 0))
    ttl_forward = optional_number(frame, find_column(frame, 'sttl'))
    ttl_backward = optional_number(frame, find_column(frame, 'dttl'))
    window_forward = optional_number(frame, find_column(frame, 'swin'))
    window_backward = optional_number(frame, find_column(frame, 'dwin'))

    row_ids = frame['orig_row_id'].astype('string')
    record_ids = [f'UNSW-NB15:{row_id}' for row_id in row_ids.astype(str)]
    features = pd.DataFrame({
        'record_id': record_ids, 'duration': duration.astype('float64'),
        'protocol': protocol.astype('string'), 'bytes_forward': forward_bytes.astype('float64'),
        'bytes_backward': backward_bytes.astype('float64'), 'service_group': service.astype('string'),
        'conn_state_group': connection_state.astype('string'),
        'packets_forward': packets_forward.astype('float64'),
        'packets_backward': packets_backward.astype('float64'),
        'mean_packet_size': mean_packet_size.astype('float64'),
        'ttl_forward': ttl_forward.astype('float64'), 'ttl_backward': ttl_backward.astype('float64'),
        'tcp_window_forward': window_forward.astype('float64'),
        'tcp_window_backward': window_backward.astype('float64'),
    })
    nsl_only_columns = ['nsl_hot', 'nsl_logged_in', 'nsl_failed_logins', 'nsl_root_shell',
                        'nsl_count_2s', 'nsl_srv_count_2s', 'nsl_same_srv_rate_2s', 'nsl_serror_rate_2s']
    for column in nsl_only_columns:
        features[column] = np.nan

    labels = pd.DataFrame({
        'record_id': record_ids, 'label_raw': raw_label.reset_index(drop=True),
        'label_source_category': category.reset_index(drop=True),
        'attack_category': attack_category,
        'label_binary': original_binary.astype('int8').reset_index(drop=True),
    })

    timestamp_column = find_column(frame, 'stime')
    if timestamp_column:
        timestamp_seconds = pd.to_numeric(frame[timestamp_column], errors='coerce')
        capture_time = pd.to_datetime(timestamp_seconds, unit='s', errors='coerce', utc=True)
    else:
        capture_time = pd.Series(pd.NaT, index=frame.index, dtype='datetime64[ns, UTC]')
    capture_time_text = capture_time.dt.strftime('%Y-%m-%dT%H:%M:%SZ').astype('string')
    duplicate_flag = (frame['_duplicate_flag'].fillna(False).astype(bool)
                      if '_duplicate_flag' in frame else pd.Series(False, index=frame.index))
    metadata = pd.DataFrame({
        'record_id': record_ids, 'source_dataset': 'UNSW-NB15',
        'orig_row_id': row_ids.to_numpy(), 'source_split': split_name,
        'capture_time': capture_time_text,
        'split_group': capture_time.dt.strftime('%Y-%m-%d').astype('string'),
        'dedup_flag': duplicate_flag.to_numpy(), 'sample_weight': 1.0,
        'has_duration': features['duration'].notna().to_numpy(),
        'invalid_duration': invalid_duration.to_numpy(),
        'has_protocol': features['protocol'].notna().to_numpy(),
        'has_directional_bytes': features[['bytes_forward', 'bytes_backward']].notna().all(axis=1).to_numpy(),
        'invalid_bytes_forward': invalid_forward.to_numpy(),
        'invalid_bytes_backward': invalid_backward.to_numpy(),
        'has_capture_time': capture_time_text.notna().to_numpy(),
        'has_service': features['service_group'].notna().to_numpy(),
        'has_connection_state': features['conn_state_group'].notna().to_numpy(),
        'has_packet_stats': features[['packets_forward', 'packets_backward', 'mean_packet_size']].notna().any(axis=1).to_numpy(),
        'has_tcp_details': features[['ttl_forward', 'ttl_backward', 'tcp_window_forward', 'tcp_window_backward']].notna().any(axis=1).to_numpy(),
        'has_nsl_content': False, 'has_nsl_window': False, 'is_kddtest21_view': False,
    })
    return features, labels, metadata, invalid_duration, invalid_forward, invalid_backward

## 4. แปลง Train/Test และตรวจ record_id

ไฟล์ cleaned ของ Train และ Test ประมวลผลแยกกัน เพื่อคง official split เดิม จากนั้นรวมตารางแต่ละชนิดตามลำดับเดียวกันและตรวจจำนวนแถว/รหัสซ้ำ

In [ ]:
features_parts, labels_parts, metadata_parts = [], [], []
quality_rows = []

for split_name, clean_path in split_files.items():
    clean_frame = pd.read_csv(clean_path)
    feature_table, label_table, metadata_table, bad_duration, bad_forward, bad_backward = (
        harmonize_split(clean_frame, split_name, clean_path.name)
    )
    features_parts.append(feature_table)
    labels_parts.append(label_table)
    metadata_parts.append(metadata_table)
    quality_rows.append({
        'source_dataset': 'UNSW-NB15', 'split': split_name,
        'rows': len(feature_table),
        'duplicate_rows': int(metadata_table['dedup_flag'].sum()),
        'invalid_duration': int(bad_duration.sum()),
        'invalid_bytes_forward': int(bad_forward.sum()),
        'invalid_bytes_backward': int(bad_backward.sum()),
    })

features = pd.concat(features_parts, ignore_index=True)
labels = pd.concat(labels_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)
if not features['record_id'].equals(labels['record_id']) or not features['record_id'].equals(metadata['record_id']):
    raise ValueError('record_id หรือการเรียงแถวของ features/labels/metadata ไม่ตรงกัน')
if features['record_id'].duplicated().any():
    raise ValueError('record_id ซ้ำ; ไม่บันทึก output ที่เชื่อมกลับแบบกำกวม')
print('จำนวนแถว:', len(features))
print('คอลัมน์ features:', len(features.columns))
display(metadata.groupby('source_split').size().rename('rows').to_frame())
display(labels.groupby(['attack_category', 'label_binary']).size().rename('rows').reset_index())

## 5. บันทึกสามตารางและรายงานตรวจสอบ

`features.csv` เก็บพฤติกรรม, `labels.csv` เก็บคำตอบ และ `metadata.csv` เก็บแหล่ง/รหัส/คุณภาพ ทั้งสามเชื่อมกันด้วย `record_id` รายงาน compatibility จะบอกภายหลังว่า feature ใดใช้ร่วมได้จริง ไม่ได้สรุปจากการตั้งชื่อกลางอย่างเดียว

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
table_rows = []
for table_name, table in [('features', features), ('labels', labels), ('metadata', metadata)]:
    table.to_csv(output_dir / f'{table_name}.csv', index=False, encoding='utf-8')
    table_rows.append({'table': table_name, 'rows': len(table), 'columns': len(table.columns)})

pd.DataFrame(table_rows).to_csv(output_dir / 'harmonization_report.csv', index=False)
pd.DataFrame(quality_rows).to_csv(output_dir / 'harmonization_quality.csv', index=False)

availability_rows = []
for split_name, table in [('train', features_parts[0]), ('test', features_parts[1])]:
    for column in table.columns:
        if column == 'record_id':
            continue
        availability_rows.append({
            'source_dataset': 'UNSW-NB15', 'split': split_name, 'feature': column,
            'rows_available': int(table[column].notna().sum()),
            'rows_total': len(table),
        })
pd.DataFrame(availability_rows).to_csv(output_dir / 'feature_availability.csv', index=False)

# ใช้ manifest ที่ตรวจ checksum แล้วใน cell แรก เป็นหลักฐานของ raw ที่สร้างตารางนี้
saved_manifest.to_csv(output_dir / 'source_manifest.csv', index=False)
provenance = {
    'dataset': 'UNSW-NB15',
    'source_version': registry['datasets']['UNSW-NB15']['version'],
    'registry_version': registry['registry_version'],
    'feature_contract_version': contract['contract_version'],
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'row_count': len(features),
    'source_files': saved_manifest.to_dict(orient='records'),
}
(output_dir / 'provenance.json').write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding='utf-8'
)

display(pd.DataFrame(table_rows))
print('สร้างไฟล์ใน:', output_dir)
print('รายการไฟล์:', ', '.join(sorted(path.name for path in output_dir.iterdir() if path.suffix.lower() == '.csv')))